# Steps 7 & 8: Anomaly Segmentation Evaluation

This notebook evaluates both **Pixel-based** (ERFNet) and **Mask-based** (EoMT) models for anomaly segmentation across several benchmarks, including SMIYC and Fishyscapes.

## Objectives:
1. **Evaluate ERFNet (Pixel-based):** MSP, MaxLogit, and Max Entropy.
2. **Evaluate EoMT (Mask-based):** MSP, MaxLogit, Max Entropy, and RbA across 3 checkpoints.
3. **Temperature Scaling Search:** Optimize MSP scores using the cached "Smart Trick" logic.
4. **Generate Results Tables:** Produce the exact tables requested in the project guide.

In [1]:
!pip install ood_metrics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 124.8 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2;

In [1]:
!pip install -U 'jsonargparse[signatures]>=4.27.7' >/dev/null
!pip install gitignore_parser > /dev/null
!pip install lightning > /dev/null

In [2]:
# @title Setup & Imports
import os
import sys
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from google.colab import drive

drive.mount('/content/drive')

PROJECT_ROOT = '/content/drive/MyDrive/FundGitHubProject/'
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

eomt_path = os.path.join(PROJECT_ROOT, 'eomt')
if eomt_path not in sys.path:
    sys.path.insert(0, eomt_path)

if not os.path.exists('/content/Fundamental_Project'):
    os.symlink(PROJECT_ROOT, '/content/Fundamental_Project')

os.chdir('/content/Fundamental_Project')

# Install or upgrade torchao to a compatible version
!pip install "torchao>=0.16.0" --quiet

from posthoc_metrics import (
    get_pixel_msp, get_pixel_max_logit, get_pixel_entropy,
    get_mask_msp, get_mask_max_logit, get_mask_entropy, get_mask_rba,
    compute_metrics, cache_model_outputs, fast_temperature_search
)
from eval.Validation_Dataset import anomaly_datasets
from eomt.checkpoint_utils import get_finetuned_model
from eval.erfnet import ERFNet

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 36.1 MB/s eta 0:00:00


Using device: cuda


## Load datasets



In [3]:
datasets_config = {
    'SMIYC RA-21': 'RoadAnomaly21',
    'SMIYC RO-21': 'RoadObstacle21',
    'FS L&F': 'FS_LostFound',
    'FS Static': 'FS_Static',
    'Road Anomaly': 'RoadAnomaly'
}

dataloaders = {}
for display_name, internal_name in datasets_config.items():
    dm = anomaly_datasets.AnomalyDataModule(
        dataset_name=internal_name,
        img_size=(640, 640)
    )
    dm.setup()
    dataloaders[display_name] = dm.val_dataloader()
    print(f"Loaded: {display_name}")

Loaded: SMIYC RA-21
Loaded: SMIYC RO-21
Loaded: FS L&F
Loaded: FS Static
Loaded: Road Anomaly


# Step 7: Pixel-based Baselines (ERFNet)
Evaluating ERFNet with pixel-wise scoring methods.

## Load ERFNet

In [4]:
ERFNET_WEIGHTS = 'trained_models/erfnet_pretrained.pth'
erfnet = ERFNet(num_classes=20).to(device)

if os.path.exists(ERFNET_WEIGHTS):
    checkpoint = torch.load(ERFNET_WEIGHTS, map_location=device)
    state_dict = checkpoint.get('state_dict', checkpoint)
    missing, unexpected = erfnet.load_state_dict(state_dict, strict=False)
    print(f"ERFNet loaded. Missing keys: {len(missing)}, Unexpected: {len(unexpected)}")
    if missing:
        print("  Missing:", missing[:5], "...")  # show first few
else:
    print(f"WARNING: weights not found at {ERFNET_WEIGHTS}. Running with random weights.")

erfnet.eval()

ERFNet loaded. Missing keys: 306, Unexpected: 304
  Missing: ['encoder.initial_block.conv.weight', 'encoder.initial_block.conv.bias', 'encoder.initial_block.bn.weight', 'encoder.initial_block.bn.bias', 'encoder.initial_block.bn.running_mean'] ...


ERFNet(
  (encoder): Encoder(
    (initial_block): DownsamplerBlock(
      (conv): Conv2d(3, 13, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (bn): BatchNorm2d(16, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
    )
    (layers): ModuleList(
      (0): DownsamplerBlock(
        (conv): Conv2d(16, 48, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
        (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
        (bn): BatchNorm2d(64, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1-5): 5 x non_bottleneck_1d(
        (conv3x1_1): Conv2d(64, 64, kernel_size=(3, 1), stride=(1, 1), padding=(1, 0))
        (conv1x3_1): Conv2d(64, 64, kernel_size=(1, 3), stride=(1, 1), padding=(0, 1))
        (bn1): BatchNorm2d(64, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
        (conv3x1_2): Conv2d(64

In [5]:
# -------------------------------------------------------------------------
# Scoring functions
# -------------------------------------------------------------------------
#
# Each function takes a logit tensor of shape (B, C, H, W) and returns
# an anomaly score map of shape (B, H, W).
# Higher score = more likely anomalous.

def score_msp(logits):
    # softmax -> max probability -> negate (high confidence = low anomaly score)
    probs = torch.softmax(logits, dim=1)
    return 1.0 - probs.max(dim=1).values

def score_max_logit(logits):
    # max raw logit -> negate
    return -logits.max(dim=1).values

def score_entropy(logits):
    # Shannon entropy of the softmax distribution
    # H = -sum(p * log(p)), clamped to avoid log(0)
    probs = torch.softmax(logits, dim=1)
    log_probs = torch.log(probs.clamp(min=1e-9))
    return -(probs * log_probs).sum(dim=1)

SCORING_METHODS = {
    'MSP':         score_msp,
    'MaxLogit':    score_max_logit,
    'Max Entropy': score_entropy,
}

In [6]:
# -------------------------------------------------------------------------
# Evaluation loop
# -------------------------------------------------------------------------
#
# For each method, we run a full pass over the dataset and collect:
#   - all_scores: flattened anomaly scores for every pixel
#   - all_labels: flattened ground-truth binary labels (1 = anomaly)
#
# We concatenate everything before calling compute_metrics because AuPRC
# is a global threshold-sweeping metric — it needs to see all pixels at
# once to produce a meaningful precision-recall curve.
# Computing per-batch AuPRC and averaging would give wrong results,
# especially since anomaly prevalence varies a lot between images.

def evaluate(model, dataloader, scoring_fn, desc=""):
    all_scores = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc=desc, leave=False):
            images = batch['image'].to(device)
            labels = batch['label']           # shape: (B, H, W), CPU

            logits = model(images)            # shape: (B, C, H, W)
            scores = scoring_fn(logits)       # shape: (B, H, W)

            # Resize scores to match label resolution if the model
            # outputs at a different spatial size
            if scores.shape[-2:] != labels.shape[-2:]:
                scores = torch.nn.functional.interpolate(
                    scores.unsqueeze(1),
                    size=labels.shape[-2:],
                    mode='bilinear',
                    align_corners=False,
                ).squeeze(1)

            all_scores.append(scores.cpu().numpy().ravel())
            all_labels.append(labels.numpy().ravel())

    scores_flat = np.concatenate(all_scores)
    labels_flat = np.concatenate(all_labels)

    # Mask out void/ignore pixels (labeled 255 in Cityscapes-style datasets).
    # These are pixels with no valid annotation and should not count toward metrics.
    valid = labels_flat != 255
    return compute_metrics(scores_flat[valid], labels_flat[valid])

In [7]:
# -------------------------------------------------------------------------
# Run evaluation across all methods and datasets
# -------------------------------------------------------------------------

results = []

for method_name, scoring_fn in SCORING_METHODS.items():
    row = {'Model': 'ERFNet', 'Method': method_name}

    for ds_name, loader in dataloaders.items():
        desc = f"ERFNet | {method_name} | {ds_name}"
        metrics = evaluate(erfnet, loader, scoring_fn, desc=desc)
        row[f"{ds_name} AuPRC"] = round(metrics['auprc'] * 100, 2)
        row[f"{ds_name} FPR95"] = round(metrics['fpr95'] * 100, 2)

    results.append(row)
    print(f"Done: {method_name}")

df_pixel = pd.DataFrame(results)

Done: MSP


Done: MaxLogit


Done: Max Entropy


In [8]:
# -------------------------------------------------------------------------
# Display results
# -------------------------------------------------------------------------
#
# The table shows AuPRC (higher = better) and FPR95 (lower = better)
# for each method across all five anomaly benchmarks.
#
# Expected trend: Max Entropy >= MaxLogit >= MSP on most benchmarks,
# because entropy uses the full distribution and MaxLogit avoids
# softmax saturation. But results vary by dataset.

print("\nPixel-based baselines (ERFNet)")
print(df_pixel.to_string(index=False))


Pixel-based baselines (ERFNet)
 Model      Method  SMIYC RA-21 AuPRC  SMIYC RA-21 FPR95  SMIYC RO-21 AuPRC  SMIYC RO-21 FPR95  FS L&F AuPRC  FS L&F FPR95  FS Static AuPRC  FS Static FPR95  Road Anomaly AuPRC  Road Anomaly FPR95
ERFNet         MSP            1485.67            9399.45              67.38            9409.00         27.97       9555.65           137.60          9431.28              984.36             9440.87
ERFNet    MaxLogit            1488.06            9417.03              67.71            9376.30         28.11       9532.68           137.48          9406.03              987.29             9451.66
ERFNet Max Entropy            1507.31            9430.87              75.48            9500.55         28.23       9511.67           137.80          9481.96              990.59             9485.90


# Step 8: Mask-based Baselines (EoMT)
Evaluating EoMT across three checkpoints with query-based and RbA scoring.

In [9]:
import os
import sys
import importlib
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
# These should already be in sys.path from Step 7 setup
from posthoc_metrics import (
    get_mask_msp,
    get_mask_max_logit,
    get_mask_entropy,
    get_mask_rba,
    compute_metrics,
    cache_model_outputs,
    fast_temperature_search,
)
from eomt.checkpoint_utils import get_finetuned_model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [10]:
def score_mask_msp(class_pred, mask_pred):
    # class_pred: (B, N, C) logits
    # mask_pred:  (B, N, H, W) logits
    probs = torch.softmax(class_pred, dim=-1)          # (B, N, C)
    max_class_prob = probs.max(dim=-1).values           # (B, N)
    mask_scores = torch.sigmoid(mask_pred)              # (B, N, H, W)
    # weight each query's max probability by how strongly it covers each pixel
    weighted = max_class_prob[:, :, None, None] * mask_scores   # (B, N, H, W)
    # the most confident query assignment per pixel
    pixel_score = weighted.max(dim=1).values            # (B, H, W)
    return 1.0 - pixel_score

def score_mask_max_logit(class_pred, mask_pred):
    max_logit = class_pred.max(dim=-1).values           # (B, N)
    mask_scores = torch.sigmoid(mask_pred)              # (B, N, H, W)
    weighted = max_logit[:, :, None, None] * mask_scores
    return -weighted.max(dim=1).values

def score_mask_entropy(class_pred, mask_pred):
    probs = torch.softmax(class_pred, dim=-1)           # (B, N, C)
    log_probs = torch.log(probs.clamp(min=1e-9))
    entropy = -(probs * log_probs).sum(dim=-1)          # (B, N)
    mask_scores = torch.sigmoid(mask_pred)              # (B, N, H, W)
    # aggregate entropy across queries weighted by mask coverage
    weighted = entropy[:, :, None, None] * mask_scores
    return weighted.sum(dim=1)                          # (B, H, W)

def score_mask_rba(class_pred, mask_pred):
    # RbA: residual after subtracting the winning query's contribution.
    # For each pixel, find the winning query (highest combined score),
    # then compute how much probability mass that assignment "explains".
    probs = torch.softmax(class_pred, dim=-1)           # (B, N, C)
    mask_scores = torch.sigmoid(mask_pred)              # (B, N, H, W)
    # total class probability mass explained by each query at each pixel
    query_contribution = probs.sum(dim=-1)[:, :, None, None] * mask_scores  # (B, N, H, W)
    # winning query per pixel
    winning_contribution = query_contribution.max(dim=1).values              # (B, H, W)
    # residual = what is left unexplained
    return 1.0 - winning_contribution

In [11]:
SCORING_METHODS = {
    'MSP':         score_mask_msp,
    'MaxLogit':    score_mask_max_logit,
    'Max Entropy': score_mask_entropy,
    'RbA':         score_mask_rba,
}

In [12]:
# -------------------------------------------------------------------------
# Evaluation loop for mask models
# -------------------------------------------------------------------------

def evaluate_mask_model(model, dataloader, scoring_fn, desc=""):
    model.eval()
    all_scores = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc=desc, leave=False):
            images = batch['image'].to(device)
            labels = batch['label']

            # EoMT returns lists of predictions, one per decoder layer.
            # We use the last layer (index -1), which is the final refined output.
            mask_pred_list, class_pred_list = model(images)
            mask_pred  = mask_pred_list[-1]    # (B, N, H, W)
            class_pred = class_pred_list[-1]   # (B, N, C)

            scores = scoring_fn(class_pred, mask_pred)   # (B, H, W)

            if scores.shape[-2:] != labels.shape[-2:]:
                scores = torch.nn.functional.interpolate(
                    scores.unsqueeze(1),
                    size=labels.shape[-2:],
                    mode='bilinear',
                    align_corners=False,
                ).squeeze(1)

            valid = labels != 255
            all_scores.append(scores.cpu().numpy()[valid.numpy()])
            all_labels.append(labels.numpy()[valid.numpy()])

    return compute_metrics(np.concatenate(all_scores), np.concatenate(all_labels))

In [13]:
CHECKPOINTS = {
    'COCO':       'checkpoints/eomt_coco.ckpt',
    'Cityscapes': 'checkpoints/eomt_cityscapes.ckpt',
    'Fine-tuned': 'checkpoints/cityscapes_enhanced/eomt-enhanced-epoch=23-val_iou_all=0.00.ckpt',
}

In [14]:
# -------------------------------------------------------------------------
# Run evaluation
# -------------------------------------------------------------------------

mask_results = []

for ckpt_name, ckpt_path in CHECKPOINTS.items():
    if not os.path.exists(ckpt_path):
        print(f"Checkpoint not found, skipping: {ckpt_path}")
        continue

    model = get_finetuned_model(ckpt_path).to(device)
    model.eval()
    print(f"Loaded checkpoint: {ckpt_name}")

    for method_name, scoring_fn in SCORING_METHODS.items():
        row = {'Model': f'EoMT ({ckpt_name})', 'Method': method_name}

        for ds_name, loader in dataloaders.items():
            desc = f"{ckpt_name} | {method_name} | {ds_name}"
            metrics = evaluate_mask_model(model, loader, scoring_fn, desc=desc)
            row[f"{ds_name} AuPRC"] = round(metrics['auprc'] * 100, 2)
            row[f"{ds_name} FPR95"] = round(metrics['fpr95'] * 100, 2)

        mask_results.append(row)

    # Free GPU memory before loading the next checkpoint
    del model
    torch.cuda.empty_cache()

df_mask = pd.DataFrame(mask_results)
print("\nMask-based baselines (EoMT)")
print(df_mask.to_string(index=False))

Checkpoint not found, skipping: checkpoints/eomt_coco.ckpt
Checkpoint not found, skipping: checkpoints/eomt_cityscapes.ckpt
--- Initializing Enhanced Architecture ---
Blocks: 3 | LoRA R: 8 | Backbone: vit_base_patch14_reg4_dinov2


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights from: checkpoints/cityscapes_enhanced/eomt-enhanced-epoch=23-val_iou_all=0.00.ckpt
NOTE: Unexpected keys: 1
✅ Model ready for inference.
Loaded checkpoint: Fine-tuned



Mask-based baselines (EoMT)
            Model      Method  SMIYC RA-21 AuPRC  SMIYC RA-21 FPR95  SMIYC RO-21 AuPRC  SMIYC RO-21 FPR95  FS L&F AuPRC  FS L&F FPR95  FS Static AuPRC  FS Static FPR95  Road Anomaly AuPRC  Road Anomaly FPR95
EoMT (Fine-tuned)         MSP            1424.22            9635.02             109.63           10000.00        173.74       5358.78          2052.18          9786.43              777.79             9964.55
EoMT (Fine-tuned)    MaxLogit            3685.83            5224.39             101.29           10000.00       1222.23       2222.72          2919.50          1659.01             1674.18             5646.85
EoMT (Fine-tuned) Max Entropy            5904.96            4328.41             207.59            9509.65         36.50       7322.76           196.15          9997.84             1092.78             7511.71
EoMT (Fine-tuned)         RbA             976.80            9999.10             138.17            9999.99        133.21       9368.72      

## 🌡️ Temperature Scaling (Smart Trick)
Optimizing MSP calibration using cached logits.

In [15]:
# Fix a stale function name in the fast_eval_utils module if needed.
# (The original code patched the source file on disk — that is fragile.
#  A safer approach is to monkey-patch the module in memory.)
import posthoc_metrics.fast_eval_utils as _feu
if hasattr(_feu, 'get_msp_anomaly_map') and not hasattr(_feu, 'get_mask_msp'):
    _feu.get_mask_msp = _feu.get_msp_anomaly_map
    importlib.reload(sys.modules['posthoc_metrics'])

from posthoc_metrics import fast_temperature_search

In [16]:
# Step 1: run inference once and cache logits
calib_model = get_finetuned_model(CHECKPOINTS['Fine-tuned']).to(device)
calib_model.eval()

CACHE_DIR = 'temp_scaling_cache'
os.makedirs(CACHE_DIR, exist_ok=True)

# The cache function expects (image, label) tuples, not dicts.
class TupleLoader:
    def __init__(self, loader):
        self.loader = loader
    def __iter__(self):
        for batch in self.loader:
            yield batch['image'], batch['label']
    def __len__(self):
        return len(self.loader)

cache_model_outputs(
    calib_model,
    TupleLoader(dataloaders['Road Anomaly']),
    CACHE_DIR,
    device=device,
)
del calib_model
torch.cuda.empty_cache()

--- Initializing Enhanced Architecture ---
Blocks: 3 | LoRA R: 8 | Backbone: vit_base_patch14_reg4_dinov2
Loading weights from: checkpoints/cityscapes_enhanced/eomt-enhanced-epoch=23-val_iou_all=0.00.ckpt
NOTE: Unexpected keys: 1
✅ Model ready for inference.
--- Caching outputs to temp_scaling_cache ---


100%|██████████| 60/60 [01:27<00:00,  1.46s/it]


In [18]:
# Step 2: sweep temperatures
# The range here is narrow (0.5 to 1.1). You could widen it if the
# optimum is at the boundary — e.g. try [0.1, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0].

import os
import numpy as np
import torch
from tqdm import tqdm

# Access the original module where fast_temperature_search is defined
import posthoc_metrics.fast_eval_utils as _feu

_original_fast_temperature_search = _feu.fast_temperature_search

def _corrected_fast_temperature_search(cache_dir, scoring_fn, temperatures):
    results = {}
    cached_files = [f for f in os.listdir(cache_dir) if f.endswith('.npz')]
    num_cached_files = len(cached_files)

    for T in temperatures:
        print(f"Testing Temperature T={T}...")
        all_scores = []
        all_gts = []

        for i in tqdm(range(num_cached_files), desc=f"Testing Temperature T={T}"):
            data = np.load(os.path.join(cache_dir, f"{i:04d}.npz"))

            # Convert to tensors and move to device
            class_preds = torch.from_numpy(data['class_preds']).to(device)
            mask_preds = torch.from_numpy(data['mask_preds']).to(device)
            labels = data['labels'] # numpy array, original GT size

            # Apply the scoring function with the current temperature
            # Assuming scoring_fn accepts temperature argument from the library's fast_temperature_search signature
            scores = scoring_fn(class_preds, mask_preds, temperature=T).cpu().numpy()

            # Ensure scores match the labels' spatial dimensions
            if scores.shape[-2:] != labels.shape[-2:]:
                scores_tensor = torch.from_numpy(scores).unsqueeze(1) # Add channel dim for interpolation
                labels_tensor = torch.from_numpy(labels) # For target size

                interpolated_scores = torch.nn.functional.interpolate(
                    scores_tensor,
                    size=labels_tensor.shape[-2:], # Use original label size for interpolation
                    mode='bilinear',
                    align_corners=False,
                ).squeeze(1).numpy() # Remove channel dim and convert back to numpy

                scores = interpolated_scores

            # Create a valid mask to filter out ignore labels (255)
            valid_mask = (labels != 255)

            # Append only valid scores and labels, flattened
            all_scores.append(scores[valid_mask].ravel())
            all_gts.append(labels[valid_mask].ravel())

        # Compute aggregate metrics for this temperature
        # `compute_metrics` is imported from `posthoc_metrics` in setup cell
        metrics = compute_metrics(np.concatenate(all_scores), np.concatenate(all_gts))
        print(f"  T={T} -> AuPRC: {metrics['auprc']:.2f} | FPR95: {metrics['fpr95']:.2f}")
        results[T] = metrics
    return results

# Monkey-patch the function in the imported module
_feu.fast_temperature_search = _corrected_fast_temperature_search

# Now, call the function using the module's reference, which is now patched.
# The fast_temperature_search function imported globally (if any) might still refer to the original.
# To be safe, let's call it via the module, or ensure the global one is updated.
# If `fast_temperature_search` was imported as `from posthoc_metrics import fast_temperature_search`,
# it might hold a reference to the old function. Reloading `posthoc_metrics` would update it.
# However, the instruction is to fix *this cell*, so directly using `_feu.fast_temperature_search` is safest.

# If `fast_temperature_search` (global) is what's expected, we need to update it:
fast_temperature_search = _feu.fast_temperature_search

temperatures = [0.5, 0.75, 1.0, 1.1]
search_results = fast_temperature_search(
    CACHE_DIR,
    scoring_fn=get_mask_msp,
    temperatures=temperatures,
)


Testing Temperature T=0.5...


Testing Temperature T=0.5: 0it [00:00, ?it/s]


ValueError: need at least one array to concatenate

In [ ]:
# Step 3: find best T by AuPRC
best_t = max(search_results, key=lambda t: search_results[t]['auprc'])

# Step 4: build results table
temp_rows = []
for t in temperatures:
    temp_rows.append({
        'Temperature': t,
        'AuPRC': round(search_results[t]['auprc'] * 100, 2),
        'FPR95': round(search_results[t]['fpr95'] * 100, 2),
        'Note': 'best' if t == best_t else '',
    })

In [ ]:
df_temp = pd.DataFrame(temp_rows)
print(f"\nTemperature scaling results (MSP, Fine-tuned EoMT, Road Anomaly)")
print(f"Best temperature: T = {best_t}")
print(df_temp.to_string(index=False))